In [1]:
from db_utilities import get_text_messages_by_id_ch_memoryEfficient
import pandas as pd
import os
from tqdm import tqdm

uri = os.environ.get('MONGO_DB_URL', 'mongodb://localhost:27017')
db_name = "Telegram_test"

In [2]:
# Extract relevant channel ids (topic)
ch_to_topic_df = pd.read_csv("data/ch_to_topic_mapping.csv")
included_topics = ["Religion", "US ews", "World news", "Extremists and radicals", "Social"]
ch_to_topic_df = ch_to_topic_df[ch_to_topic_df.topic.isin(included_topics)]
topic_relevant_ids = ch_to_topic_df.ch_ID.to_list()

In [3]:
# Extract relevant channel ids (language)
ch_to_language_df = pd.read_csv('data/channel_to_language_mapping.csv', sep="\t")
ch_to_language_df = ch_to_language_df[ch_to_language_df["language"] == "en"]
language_relevant_ids = ch_to_language_df.ch_id.to_list()

In [4]:
# Combin ids to identify all relevant channels
relevant_ch_ids = list(set(language_relevant_ids + topic_relevant_ids))
len(relevant_ch_ids)

19768

In [ ]:
# For validation afterwards
with open("data/relevant_ids.txt", "w") as file:
    file.write(str(relevant_ch_ids))

# Extract all messages for each channel

In [23]:
def write_mongodb_to_parquet(ids):
    missing_ids = []
    batch_size = int(len(ids) / 5) + 1

    for i in range(0, len(ids), batch_size):
        batch_ids = ids[i:i+batch_size]
    
        messages_dict = {"id": [], "message":[]}
        
        # Iterate over each channel
        for id in tqdm(batch_ids):
            try:
                msgs = get_text_messages_by_id_ch_memoryEfficient(id_channel=id)
                # Iterate over each message
                for msg in msgs.values():
                    messages_dict["message"].append(msg["message"])
                    messages_dict["id"].append(id)
            # Append ids that are not in the database (file 75 and 106 broken)
            except:
                missing_ids.append(id)

                
        all_messages_df = pd.DataFrame(messages_dict)
        all_messages_df.to_parquet(f"data/batches/messages_{i+batch_size}.parquet")
    
    return missing_ids
    
  

In [24]:
missing_ids = write_mongodb_to_parquet(relevant_ch_ids)

100%|██████████| 3952/3952 [1:28:00<00:00,  1.34s/it]    


In [25]:
with open("data/missing_ids.txt", "w") as f:
    f.write(str(missing_ids))